# 🤖 Agentic AI Workshop – Building an AI Agent

## Introduction

This project demonstrates the basic working of an Agentic AI system using
Google Colab and the Groq API.

Unlike a simple chatbot, an AI agent can decide what action is required,
use external tools, observe the results, and continue working until the
given task is completed.

The agent in this project follows the basic agent loop:

**THINK → ACT → OBSERVE → REPEAT → FINAL ANSWER**

## Objective

The main objectives of this project are:

- Understand the basic concept of Agentic AI.
- Connect an AI model using the Groq API.
- Create and use tools with an AI agent.
- Demonstrate the THINK, ACT, OBSERVE and REPEAT process.
- Build an agent that can select between different tools.
- Test the agent with different tasks.

## Tools Used

- Google Colab
- Python
- Groq API
- Groq Python SDK
- Calculator Tool
- Word Counter Tool

In [2]:
!pip install -q groq

In [4]:
from google.colab import userdata
from groq import Groq

api_key = userdata.get("GROQ_API_KEY")

client = Groq(api_key=api_key)

print("Groq API key connected successfully!")

Groq API key connected successfully!


In [6]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get("GROQ_API_KEY"))

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "What is Agentic AI? Explain in 2 simple sentences."
        }
    ]
)

print(response.choices[0].message.content)

Agentic AI refers to artificial‑intelligence systems designed to act autonomously toward goals, making decisions and taking actions without direct human instruction for each step. It combines sophisticated perception, planning, and self‑adjustment so the AI can pursue objectives like a self‑directed agent in dynamic environments.


In [7]:
from google.colab import userdata
from groq import Groq

# Connect to Groq using the API key stored in Colab Secrets
client = Groq(api_key=userdata.get("GROQ_API_KEY"))

MODEL = "openai/gpt-oss-120b"

def ask_agent(goal):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": """
                You are a simple AI agent.

                Follow this process:
                1. THINK - understand what needs to be done.
                2. ACT - decide what action should be taken.
                3. OBSERVE - consider the result.
                4. REPEAT - continue if more work is needed.
                5. FINAL - give the final answer.

                Keep your explanation simple and clear.
                """
            },
            {
                "role": "user",
                "content": goal
            }
        ]
    )

    return response.choices[0].message.content


# Test the agent
goal = "Explain how cloud computing helps students."
answer = ask_agent(goal)

print(answer)

**THINK** – What do students need?  
- Easy access to software and files from any device.  
- Low‑cost tools for learning, collaboration, and projects.  
- Reliable storage that won’t be lost if a laptop breaks.  

**ACT** – Show how cloud computing meets those needs.  

| Student Need | Cloud Computing Benefit |
|--------------|--------------------------|
| **Access anywhere** | Applications (Google Docs, Microsoft 365, coding IDEs) run in a web browser, so a student can work from a laptop, tablet, or phone on campus, at home, or on the go. |
| **Collaboration** | Multiple users can edit the same document, spreadsheet, or presentation in real time, see each other’s changes, comment, and chat without emailing files back and forth. |
| **Cost savings** | Most cloud services offer free tiers or subscription plans that are cheaper than buying expensive software licenses or high‑end hardware. |
| **Scalable resources** | When a project needs more computing power (e.g., data analysis, 3‑D r

In [9]:
def calculator(expression):
    """Calculate a mathematical expression."""
    try:
        result = eval(expression)
        return str(result)
    except:
        return "Invalid mathematical expression"


# Test the calculator
print(calculator("22 * 4"))

88


In [14]:
import json

# Describe the calculator tool to the AI
calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Calculate a mathematical expression.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "The mathematical expression to calculate."
                }
            },
            "required": ["expression"]
        }
    }
}


# Connect the tool name to the actual Python function
available_functions = {
    "calculator": calculator
}


def run_agent(user_question):

    messages = [
        {
            "role": "system",
            "content": (
                "You are an AI agent. "
                "Use the calculator tool whenever a mathematical calculation "
                "is required. After receiving the tool result, give the final answer."
            )
        },
        {
            "role": "user",
            "content": user_question
        }
    ]

    # Ask the AI whether it needs the calculator
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=[calculator_tool],
        tool_choice="auto"
    )

    assistant_message = response.choices[0].message
    messages.append(assistant_message)

    # Check whether the AI requested a tool
    if assistant_message.tool_calls:

        for tool_call in assistant_message.tool_calls:

            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            # Execute the requested tool
            result = available_functions[function_name](**arguments)

            print("ACT → Calculator used")
            print("OBSERVE → Calculator result:", result)

            # Send the tool result back to the AI
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": str(result)
            })

        # Ask the AI for the final answer
        final_response = client.chat.completions.create(
            model=MODEL,
            messages=messages
        )

        return final_response.choices[0].message.content

    else:
        return assistant_message.content


# Test the agent
question = "What is 125 * 8 + 50?"

print("THINK → The agent decides whether a tool is needed.")
print()

answer = run_agent(question)

print()
print("FINAL ANSWER →", answer)

THINK → The agent decides whether a tool is needed.

ACT → Calculator used
OBSERVE → Calculator result: 1050

FINAL ANSWER → The result of \(125 \times 8 + 50\) is **1050**.


In [15]:
def run_agent_loop(user_question, max_iterations=5):

    messages = [
        {
            "role": "system",
            "content": (
                "You are an AI agent. Solve the user's task step by step. "
                "Use the calculator tool whenever a calculation is required. "
                "After receiving a tool result, decide whether another action "
                "is needed. Continue until the task is complete."
            )
        },
        {
            "role": "user",
            "content": user_question
        }
    ]

    for iteration in range(max_iterations):

        print(f"\n--- ITERATION {iteration + 1} ---")
        print("THINK → Agent is deciding what to do...")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=[calculator_tool],
            tool_choice="auto"
        )

        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        # If no tool is requested, the agent is finished
        if not assistant_message.tool_calls:
            print("FINAL → Agent has completed the task.")
            return assistant_message.content

        # Execute requested tools
        for tool_call in assistant_message.tool_calls:

            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            print(f"ACT → Using {function_name}")
            print(f"Arguments → {arguments}")

            result = available_functions[function_name](**arguments)

            print(f"OBSERVE → Result: {result}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": str(result)
            })

    return "Agent stopped because the maximum number of iterations was reached."

In [16]:
question = """
Calculate the following:
First calculate 125 * 8.
Then add 250 to the result.
"""

answer = run_agent_loop(question)

print("\nFINAL ANSWER:")
print(answer)


--- ITERATION 1 ---
THINK → Agent is deciding what to do...
FINAL → Agent has completed the task.

FINAL ANSWER:
First, \(125 \times 8 = 1000\).

Next, add 250:

\(1000 + 250 = 1250\).

**Result: 1250**.


In [17]:
def word_counter(text):
    """Count the number of words in a text."""
    words = text.split()
    return len(words)


# Test the tool
text = "Agentic AI can think, act, observe and repeat."
print("Word count:", word_counter(text))

Word count: 8


In [18]:
# Add both tools to the agent

available_functions = {
    "calculator": calculator,
    "word_counter": word_counter
}


# Tell the AI about the word counter tool

word_counter_tool = {
    "type": "function",
    "function": {
        "name": "word_counter",
        "description": "Count the number of words in a given text.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string",
                    "description": "The text whose words should be counted."
                }
            },
            "required": ["text"]
        }
    }
}


# Create a list containing BOTH tools

all_tools = [
    calculator_tool,
    word_counter_tool
]

print("Two tools are ready:")
print("1. Calculator")
print("2. Word Counter")

Two tools are ready:
1. Calculator
2. Word Counter


In [21]:
# ============================================
# FINAL MULTI-TOOL AGENT
# ============================================

def run_multi_tool_agent(user_question, max_iterations=5):

    messages = [
        {
            "role": "system",
            "content": """
            You are an Agentic AI assistant.

            Follow this process:
            1. THINK - understand the user's request.
            2. ACT - choose the correct tool when needed.
            3. OBSERVE - examine the tool result.
            4. REPEAT - use another tool if more work is required.
            5. FINAL - provide the final answer.

            Available tools:
            - Calculator: for mathematical calculations.
            - Word Counter: for counting words.

            Use tools only when they are useful.
            """
        },
        {
            "role": "user",
            "content": user_question
        }
    ]

    # Repeat the agent process
    for iteration in range(max_iterations):

        print(f"\n========== ITERATION {iteration + 1} ==========")
        print("THINK → Agent is deciding what to do...")

        # Ask the AI what to do
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=all_tools,
            tool_choice="auto"
        )

        assistant_message = response.choices[0].message

        # Add AI response to conversation
        messages.append(assistant_message)

        # If the AI does not need a tool, finish
        if not assistant_message.tool_calls:

            print("FINAL → No more tools are required.")
            return assistant_message.content

        # Execute every tool requested by the AI
        for tool_call in assistant_message.tool_calls:

            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            print(f"ACT → Using tool: {function_name}")
            print(f"Arguments → {arguments}")

            # Find and execute the correct function
            if function_name in available_functions:

                result = available_functions[function_name](**arguments)

                print(f"OBSERVE → Tool result: {result}")

            else:

                result = "Tool not found."
                print("OBSERVE → Tool not found.")

            # Send the tool result back to the AI
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": str(result)
            })

    return "The agent stopped after reaching the maximum number of iterations."

In [23]:
question = """
Calculate 35 * 20.
Then count the number of words in this sentence:
"Agentic AI can think act observe and repeat."
"""

answer = run_multi_tool_agent(question)

print("\n================================")
print("FINAL ANSWER")
print("================================")
print(answer)


========== ITERATION 1 ==========
THINK → Agent is deciding what to do...
ACT → Using tool: calculator
Arguments → {'expression': '35 * 20'}
OBSERVE → Tool result: 700

========== ITERATION 2 ==========
THINK → Agent is deciding what to do...
ACT → Using tool: word_counter
Arguments → {'text': 'Agentic AI can think act observe and repeat.'}
OBSERVE → Tool result: 8

========== ITERATION 3 ==========
THINK → Agent is deciding what to do...
FINAL → No more tools are required.

FINAL ANSWER
The product of 35 × 20 is **700**.

The sentence “Agentic AI can think act observe and repeat.” contains **8 words**.


## How the Agent Works

The agent follows these steps:

1. **THINK** – The AI understands the user's request and decides what
   action is required.

2. **ACT** – The AI selects and calls an appropriate tool.

3. **OBSERVE** – The agent receives and examines the result produced by
   the tool.

4. **REPEAT** – If the task is not complete, the agent can perform
   another action.

5. **FINAL ANSWER** – When the task is complete, the agent provides the
   final response.

### Agent Workflow

User Request
↓
THINK
↓
Select Tool
↓
ACT
↓
Tool Result
↓
OBSERVE
↓
REPEAT if required
↓
FINAL ANSWER

In [24]:
# ============================================
# TEST CASE 1 - CALCULATOR
# ============================================

question = "Calculate 45 * 12 + 30."

answer = run_multi_tool_agent(question)

print("\n================================")
print("TEST CASE 1 - FINAL ANSWER")
print("================================")
print(answer)


========== ITERATION 1 ==========
THINK → Agent is deciding what to do...
ACT → Using tool: calculator
Arguments → {'expression': '45 * 12 + 30'}
OBSERVE → Tool result: 570

========== ITERATION 2 ==========
THINK → Agent is deciding what to do...
FINAL → No more tools are required.

TEST CASE 1 - FINAL ANSWER
The result of the calculation \(45 \times 12 + 30\) is **570**.


In [25]:
# ============================================
# TEST CASE 2 - WORD COUNTER
# ============================================

question = """
Count the number of words in this sentence:
Agentic AI can think act observe and repeat.
"""

answer = run_multi_tool_agent(question)

print("\n================================")
print("TEST CASE 2 - FINAL ANSWER")
print("================================")
print(answer)


========== ITERATION 1 ==========
THINK → Agent is deciding what to do...
ACT → Using tool: word_counter
Arguments → {'text': 'Agentic AI can think act observe and repeat.'}
OBSERVE → Tool result: 8

========== ITERATION 2 ==========
THINK → Agent is deciding what to do...
FINAL → No more tools are required.

TEST CASE 2 - FINAL ANSWER
The sentence contains **8 words**.


In [26]:
# ============================================
# TEST CASE 3 - MULTI-TOOL AGENT
# ============================================

question = """
First calculate 25 * 16.
Then count the number of words in this sentence:
Agentic AI can think act observe and repeat.
"""

answer = run_multi_tool_agent(question)

print("\n================================")
print("TEST CASE 3 - FINAL ANSWER")
print("================================")
print(answer)


========== ITERATION 1 ==========
THINK → Agent is deciding what to do...
ACT → Using tool: calculator
Arguments → {'expression': '25 * 16'}
OBSERVE → Tool result: 400

========== ITERATION 2 ==========
THINK → Agent is deciding what to do...
ACT → Using tool: word_counter
Arguments → {'text': 'Agentic AI can think act observe and repeat.'}
OBSERVE → Tool result: 8

========== ITERATION 3 ==========
THINK → Agent is deciding what to do...
FINAL → No more tools are required.

TEST CASE 3 - FINAL ANSWER
The calculation gives **400**, and the sentence contains **8 words**.


## Conclusion

This project successfully demonstrates a simple Agentic AI system using
Groq and Python in Google Colab.

The agent can understand a user's request, decide which tool is required,
use the appropriate tool, observe the result, and provide a final answer.

The project demonstrates the basic Agentic AI workflow:

**THINK → ACT → OBSERVE → REPEAT → FINAL ANSWER**

Two tools were implemented in this project:

- Calculator Tool
- Word Counter Tool

The multi-tool test also demonstrated that the agent can use more than one
tool to complete a single task.